# Calibrated Explanations: Dashboard Visualization Plugin Example

This notebook demonstrates the dashboard visualization plugin. It assembles
multiple registered plot plugins and an optional narrative into plotly figures.

##  parameter

| Value | Behaviour |
|---|---|
|  (default) | One dashboard figure per instance. Narrative is per-instance. |
|  | One dashboard for the whole collection. Narrative covers all instances. |

Select a specific instance with CE's own indexing:  rather
than an  option.

**Prerequisites:**

- 

Optionally, to include SHAP panels:

- 
- 

In [1]:
import os

import calibrated_explanations.plugins.registry as registry

DASHBOARD_STYLE_ID = "official.visualization.dashboard"
CE_DEFAULT_STYLE_ID = "official.visualization.dashboard.ce_default"

# Trust and load the dashboard plugin.
os.environ["CE_TRUST_PLUGIN"] = ",".join([
    DASHBOARD_STYLE_ID,
    "official.visualization.dashboard.builder",
    "official.visualization.dashboard.renderer",
    CE_DEFAULT_STYLE_ID,
    "official.visualization.dashboard.ce_default.builder",
    "official.visualization.dashboard.ce_default.renderer",
])
registry.load_entrypoint_plugins(include_untrusted=False)

print("Dashboard style loaded:    ", registry.find_plot_style_descriptor(DASHBOARD_STYLE_ID) is not None)
print("CE default style loaded:   ", registry.find_plot_style_descriptor(CE_DEFAULT_STYLE_ID) is not None)

Dashboard style loaded:     True
CE default style loaded:    True


C:\Users\loftuw\CUDATemp\ipykernel_28380\2446689546.py:17: UserWarning: Skipping untrusted plugin 'official.explanation.factual.shap' from official discovered via entrypoint. Set CE_TRUST_PLUGIN, add it to [tool.calibrated_explanations.plugins].trusted, or call trust_plugin('official.explanation.factual.shap') to load it.
  registry.load_entrypoint_plugins(include_untrusted=False)
C:\Users\loftuw\CUDATemp\ipykernel_28380\2446689546.py:17: UserWarning: Skipping untrusted plugin 'official.visualization.dashboard.bootstrap' from official discovered via entrypoint. Set CE_TRUST_PLUGIN, add it to [tool.calibrated_explanations.plugins].trusted, or call trust_plugin('official.visualization.dashboard.bootstrap') to load it.
  registry.load_entrypoint_plugins(include_untrusted=False)
C:\Users\loftuw\CUDATemp\ipykernel_28380\2446689546.py:17: UserWarning: Skipping untrusted plugin 'official.visualization.factual.shap.bootstrap' from official discovered via entrypoint. Set CE_TRUST_PLUGIN, add it

## Fit, Calibrate, and Explain

Use the normal `WrapCalibratedExplainer` workflow.

In [2]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

from calibrated_explanations import WrapCalibratedExplainer

X, y = make_classification(
    n_samples=240,
    n_features=6,
    n_informative=4,
    n_redundant=0,
    random_state=0,
)

X_train, X_test, y_train, _ = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)
X_fit, X_cal, y_fit, y_cal = train_test_split(
    X_train, y_train, test_size=0.25, random_state=0, stratify=y_train
)

model = WrapCalibratedExplainer(LogisticRegression(random_state=0, solver="liblinear"))
model.fit(X_fit, y_fit)
model.calibrate(X_cal, y_cal, mode="classification")

factual_explanations = model.explain_factual(X_test[:4])
alternative_explanations = model.explore_alternatives(X_test[:4])
print("Factual explanations:", len(factual_explanations.explanations))
print("Alternative explanations:", len(alternative_explanations.explanations))

Factual explanations: 4
Alternative explanations: 4


## Factual Dashboard — Per Instance (default)

Pass the full collection. With  (default) you get one dashboard
figure per instance.  is the first; the rest are in
.

To get a single instance, use CE’s indexing: .

In [ ]:
# Default: one dashboard per instance.
result = factual_explanations.plot(
    style=DASHBOARD_STYLE_ID,
    show=True,
    plots=[
        {"style": CE_DEFAULT_STYLE_ID},
    ],
    narrative=True,
    expertise_level="beginner",
    title="Factual Explanation Dashboard",
)
# result.figure is instance 0; result.extras["extra_figures"] holds the rest.
print("Instances rendered:", result.extras["n_instances"])
result.figure

C:\Users\loftuw\Documents\Github\kristinebergs-calibrated_explanations\src\calibrated_explanations\explanations\explanation.py:668: UserWarning:

Narrative template fallback: default template used because provided relative path was missing

C:\Users\loftuw\Documents\Github\kristinebergs-calibrated_explanations\src\calibrated_explanations\explanations\explanation.py:668: UserWarning:

Narrative template fallback: default template used because provided relative path was missing

C:\Users\loftuw\Documents\Github\kristinebergs-calibrated_explanations\src\calibrated_explanations\explanations\explanation.py:668: UserWarning:

Narrative template fallback: default template used because provided relative path was missing



Instances rendered: 4


C:\Users\loftuw\Documents\Github\kristinebergs-calibrated_explanations\src\calibrated_explanations\explanations\explanation.py:668: UserWarning:

Narrative template fallback: default template used because provided relative path was missing



<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

In [ ]:
result.extras["extra_figures"][1]

## Alternative Dashboard — Per Instance (default)

Same pattern for alternative explanations.

In [14]:
result = alternative_explanations.plot(
    style=DASHBOARD_STYLE_ID,
    show=True,
    plots=[
        {
            "style": CE_DEFAULT_STYLE_ID,
            "filter_top": 10,
        },
        {
            "style": CE_DEFAULT_STYLE_ID,
            "ce_style": "ensured",    # forwarded as style= to explanation.plot()
        },
    ],
    narrative=False,
)
print("Instances rendered:", result.extras["n_instances"])
result.figure

Instances rendered: 4


<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

## Combined Dashboard — All Instances

Set  to get a single dashboard for the whole collection.
The narrative is generated for the collection (all instances), not just the first.

In [5]:
# per_instance=False: one dashboard for all instances, narrative covers the collection.
result = factual_explanations.plot(
    style=DASHBOARD_STYLE_ID,
    show=True,
    per_instance=False,
    plots=[
        {"style": CE_DEFAULT_STYLE_ID},
    ],
    narrative=True,
    expertise_level="intermediate",
    title="Combined Factual Dashboard — All Instances",
)
result.figure

C:\Users\loftuw\Documents\Github\kristinebergs-calibrated_explanations\src\calibrated_explanations\explanations\explanation.py:668: UserWarning:

Narrative template fallback: default template used because provided relative path was missing

C:\Users\loftuw\Documents\Github\kristinebergs-calibrated_explanations\src\calibrated_explanations\explanations\explanation.py:668: UserWarning:

Narrative template fallback: default template used because provided relative path was missing

C:\Users\loftuw\Documents\Github\kristinebergs-calibrated_explanations\src\calibrated_explanations\explanations\explanation.py:668: UserWarning:

Narrative template fallback: default template used because provided relative path was missing

C:\Users\loftuw\Documents\Github\kristinebergs-calibrated_explanations\src\calibrated_explanations\explanations\explanation.py:668: UserWarning:

Narrative template fallback: default template used because provided relative path was missing



<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

## Dashboard with SHAP Panel (optional)

If `calibrated-explanations-explanation-factual-shap` and
`calibrated-explanations-visualization-factual-shap` are installed, you can mix
a CE default panel with a SHAP panel in the same dashboard.

In [6]:
EXPLANATION_ID = "official.explanation.factual.shap"
SHAP_STYLE_ID  = "official.visualization.factual.shap"

shap_available = (
    registry.find_explanation_descriptor(EXPLANATION_ID) is not None
    and registry.find_plot_style_descriptor(SHAP_STYLE_ID) is not None
)

if shap_available:
    shap_model = WrapCalibratedExplainer(LogisticRegression(random_state=0, solver="liblinear"))
    shap_model.fit(X_fit, y_fit)
    shap_model.calibrate(
        X_cal, y_cal, mode="classification", factual_plugin=EXPLANATION_ID
    )
    shap_explanations = shap_model.explain_factual(X_test[:2])

    result = shap_explanations[0].plot(
        style=DASHBOARD_STYLE_ID,
        show=True,
        plots=[
            {"style": CE_DEFAULT_STYLE_ID},
            {
                "style": SHAP_STYLE_ID,
                "shap_kind": "waterfall",
                "shap_bound": "center",
            },
        ],
        narrative=True,
        expertise_level="beginner",
        title="CE Default + SHAP Waterfall Dashboard",
    )
    display(result.figure)
else:
    print("SHAP plugins not loaded â€” install and trust them to run this cell.")

C:\Users\loftuw\Documents\Github\kristinebergs-calibrated_explanations\src\calibrated_explanations\core\explain\orchestrator.py:1790: UserWarning:

Using untrusted explanation plugin 'official.explanation.factual.shap' via explicit override. Ensure you trust the source of this plugin.

ExactExplainer explainer: 3it [00:12,  6.13s/it]               


TypeError: calibrated_explanations.plotting.plot_probabilistic() got multiple values for keyword argument 'title'

## Save to HTML

The dashboard saves as an interactive HTML file by default.

In [ ]:
# per_instance=True (default): saves one file per instance, suffixed _0, _1, ...
saved_result = factual_explanations.plot(
    style=DASHBOARD_STYLE_ID,
    show=False,
    path="dashboard",
    save_ext=".html",
    plots=[
        {"style": CE_DEFAULT_STYLE_ID},
    ],
    narrative=True,
    title="Saved Dashboard",
)
print("Saved to:", saved_result.saved_paths)

C:\Users\loftuw\Documents\Github\kristinebergs-calibrated_explanations\src\calibrated_explanations\explanations\explanation.py:668: UserWarning:

Narrative template fallback: default template used because provided relative path was missing

C:\Users\loftuw\Documents\Github\kristinebergs-calibrated_explanations\src\calibrated_explanations\explanations\explanation.py:668: UserWarning:

Narrative template fallback: default template used because provided relative path was missing

C:\Users\loftuw\Documents\Github\kristinebergs-calibrated_explanations\src\calibrated_explanations\explanations\explanation.py:668: UserWarning:

Narrative template fallback: default template used because provided relative path was missing



Saved to: ()


C:\Users\loftuw\Documents\Github\kristinebergs-calibrated_explanations\src\calibrated_explanations\explanations\explanation.py:668: UserWarning:

Narrative template fallback: default template used because provided relative path was missing

